# Queen Editor — Colab kurulumu

Bu defterin işi kurmak ve sunmak: Drive'ı bağlar → repoyu klonlar → **ComfyUI'yi kurar** (19 custom
node) → **Flask** arayüzü servis eder → **cloudflared** linki basar. Üretim uygulamanın içinde
oluyor: üreticileri — model dosyalarını da ses motorunu da — arayüzdeki **Üreticiler** panelinden
kurarsın, sonra bir projeye girip kareler üretirsin. Her kare foto ile başlar, üstüne video ve ses
katmanı alabilir; hepsi `MyDrive/queenEditor/<proje>/` altına düşer.

> **Runtime → Change runtime type → T4 GPU** gerekiyor (SDXL). CPU runtime'da kurulum hücresi durur.

## Kullanım
1. Bu `app.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. **🔑 Secrets** panelinde iki secret olmalı: `GITHUB_TOKEN` (fine-grained, yalnız bu repo,
   `Contents: read`) ve `CIVITAI_COOKIE` (civitai.red → giriş yap → F12 → Application → Cookies →
   `__Secure-civ-token` değeri; ~30 günde bir yenilenir). Video üreteceksen üçüncüsü:
   `XAI_API_KEY` — video prompt'unu yazan dil modeli için; yoksa foto üretimi yine çalışır.
3. **Runtime → Run all** → Drive izni ver → ilk kurulum ~5-10 dk → en alttaki linke gir →
   **Üreticiler** panelinden kurmak istediğin üreticiyi kur.

In [ ]:
# === CONFIG ===
# The GitHub token comes from Colab's Secrets store (🔑 in the left sidebar), NOT this cell
# -- set once per Google account, never pasted again, never in the notebook source or git.
# Add a secret named GITHUB_TOKEN (fine-grained, this repo, "Contents: read") and grant this
# notebook access. See README for the token setup.
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""   # secret missing or access not granted -> the assert below explains the fix

BRANCH       = "feat/queen-editor-v3"       # dev branch for now; switch to "main" after merge
REPO         = "AltanBaysal/Internal-tools" # <owner>/<repo>
CLONE_DIR    = "/content/Internal-tools"    # clone target on Colab's local disk
APP_PORT     = 8000                         # Flask port (matches backend/config.py)
DRIVE_FOLDER = "queenEditor"                # proje kökü (MyDrive altında) — adı buradan değiştir

# === ComfyUI (kurulum + üretim; backend QE_COMFY_URL ile bu adrese konuşur) ===
COMFY_PORT  = 8188
COMFY_ROOT  = "/content/ComfyUI"
COMFY_LOG   = "/content/comfyui.log"
COMFYUI_URL = f"http://127.0.0.1:{COMFY_PORT}"

# Civitai's gated models need the session cookie. This notebook downloads nothing -- the cookie is
# passed to the app, which installs those files from its own Üreticiler panel. Like GITHUB_TOKEN it
# comes from Colab Secrets: this notebook is committed, so a pasted session JWT would land in git.
try:
    COOKIE_VALUE = userdata.get("CIVITAI_COOKIE")
except Exception:
    COOKIE_VALUE = ""

# A video's prompt is written by xAI when the job's turn comes; the key comes from Secrets like the
# two above. No assert: a photo-only run needs no language model, and stopping the notebook over a
# key that run never uses would be wrong.
try:
    XAI_API_KEY = userdata.get("XAI_API_KEY")
except Exception:
    XAI_API_KEY = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu notebook'a erişimi aç (fine-grained, yalnız bu repo, Contents: read)."
)
# Asserted here rather than at the first install: hearing it now costs a second, hearing it after
# pressing Kur costs the whole setup run.
assert len(COOKIE_VALUE or "") > 200, (
    "❌ CIVITAI_COOKIE yok/çok kısa — Colab 🔑 Secrets'a 'CIVITAI_COOKIE' adıyla ekle: "
    "civitai.red → giriş → F12 → Application → Cookies → __Secure-civ-token değeri (ES256 JWT)"
)

# SDXL needs a GPU. A CPU runtime has no driver at all, so nvidia-smi is missing rather than
# failing -- and ComfyUI would come up fine there and then fail on every render.
import subprocess as _sp
try:
    _gpu = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    _gpu_name = _gpu.stdout.strip() if _gpu.returncode == 0 else ""
except FileNotFoundError:
    _gpu_name = ""
assert _gpu_name, (
    "❌ GPU yok — Runtime → Change runtime type → T4 GPU seç ve Run all'ı yeniden çalıştır"
)

print(f"✓ GPU: {_gpu_name}")
print("✓ CONFIG hazır (token Colab Secrets'tan okundu)")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")
print(f"✓ Proje kökü: MyDrive/{DRIVE_FOLDER}")
print(f"✓ xAI anahtarı: {'okundu' if XAI_API_KEY else 'yok — video prompt yazılamaz'}")

In [ ]:
# === Mount Google Drive ===
# Projects ARE Drive folders, so the mount must succeed before the server starts: writing under
# /content/drive without a mount silently lands on Colab's local disk, and those folders die with
# the runtime. The first run opens a Google permission window -- grant it.
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # first run creates it; later runs reuse it
assert os.path.isdir(DRIVE_ROOT), f"❌ Proje kökü oluşmadı: {DRIVE_ROOT}"
print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")

In [ ]:
# === Clone (delete-and-reclone: the local tree is disposable, always fetch the latest) ===
# subprocess.run with an argument LIST (not shell=True): the token never reaches the shell
# history or a log line. On failure git's stderr is printed RAW, with the token masked.
import os, shutil, subprocess

def _mask(text):
    """Replace the token with <token> so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)              # no pull/merge -- a fresh clone has one behaviour

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"   # never printed (carries the token)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Raw git output, token masked -- never invent a cause (repo comment rule).
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The built frontend ships in the repo (frontend/dist) -- fail loud if it is missing, so a
# forgotten rebuild-and-commit shows up here, not as a blank page.
DIST = os.path.join(CLONE_DIR, "queen-editor", "frontend", "dist", "index.html")
assert os.path.exists(DIST), f"❌ Derlenmiş arayüz yok: {DIST} — frontend'i derleyip commit'le (README)"

# Both graphs ship with the repo (our own copies) -- a forgotten commit shows up here rather than
# at the first render. Sound has no graph: it runs in the app's own process.
for _name in ("workflow_api.json", "workflow_video_api.json"):
    _path = os.path.join(CLONE_DIR, "queen-editor", _name)
    assert os.path.exists(_path), f"❌ Grafik yok: {_path} — {_name} commit'lenmiş mi?"
print("✓ Klon tamam (derlenmiş arayüz + iki grafik mevcut)")

In [ ]:
# === Shared helpers — log + fail-loud run ===
# Used by the custom node cell below; kept as its own cell so the install cell stays about what it
# installs.
import time, subprocess

# The cell below does not need CONFIG (it installs to hardcoded local paths, no Drive), so without
# this gate a failed CONFIG cell stays invisible until the app starts -- after a ~10 min install.
assert "COMFY_ROOT" in globals(), "❌ Önce 1) CONFIG hücresini çalıştır"

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for install failures: git and pip both exit non-zero and say why, and that
    sentence is what gets raised rather than a guess at what went wrong.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

print("✓ Ortak yardımcılar hazır (log, run)")

## ComfyUI + Custom Node'lar (19)

İki grafiğin ihtiyacı olan paketler + Manager: ilk sekizi foto grafiği, kalanı video grafiği için.
Liste grafiklerin node künyelerinden çıkarıldı; kalan node'lar comfy-core, kurulum istemez. Biri
başarısız olursa hücre `RuntimeError` ile durur (fail-loud).

> **Ses burada yok.** Ses ComfyUI'de üretilmiyor — motoru uygulamanın kendi sürecinde çalışıyor ve
> arayüzdeki **Üreticiler** panelinden kuruluyor, tıpkı modeller gibi.

In [ ]:
%cd /content

# === System deps + ComfyUI ===
# ffmpeg is the app's own tool, not ComfyUI's: the export joins the videos with it and a sound job
# cuts the video it reads with it.
!apt-get install -y aria2 ffmpeg > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python imageio imageio-ffmpeg

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides to a graph this app ships.
# Two graphs, one ComfyUI: the photo graph needs the first block, the video graph the second.
CUSTOM_NODES = [
    ("ComfyUI-Manager",           "https://github.com/ltdrdata/ComfyUI-Manager.git"),          # detect missing nodes in the UI
    ("rgthree-comfy",             "https://github.com/rgthree/rgthree-comfy.git"),             # Power Lora Loader, Seed, Fast Groups Bypasser, Image Comparer
    ("ComfyUI-Impact-Pack",       "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),      # FaceDetailer, wildcard prompts, SAMLoader, switches
    ("ComfyUI-Impact-Subpack",    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git"),   # UltralyticsDetectorProvider
    ("ComfyUI-Easy-Use",          "https://github.com/yolain/ComfyUI-Easy-Use.git"),           # easy int/float, easy hiresFix, easy cleanGpuUsed
    ("ComfyUI-Custom-Scripts",    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),  # MathExpression
    ("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git"),   # UltimateSDUpscale (tiled Remacri upscale)
    ("ComfyUI-KJNodes",           "https://github.com/kijai/ComfyUI-KJNodes.git"),             # ImageResizeKJv2, ColorMatch
    # --- the video graph (workflow_video_api.json) ---
    ("comfy_mtb",                 "https://github.com/melMass/comfy_mtb.git"),                 # Pick From Batch, RIFEInterpolation
    ("ComfyUI-VideoHelperSuite",  "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),  # VHS_VideoCombine (the mp4 comes out here)
    ("ComfyUI-WanVideoWrapper",   "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),     # Wan video nodes
    ("ComfyUI-GGUF",              "https://github.com/city96/ComfyUI-GGUF.git"),               # UnetLoaderGGUF -- unreachable in the graph, but ComfyUI validates every node it is sent
    ("ComfyMath",                 "https://github.com/evanspearman/ComfyMath.git"),            # ComfyMathExpression (seconds -> frames)
    ("ComfyUI-Frame-Interpolation", "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"),  # RIFE
    ("ComfyUI-VFI",               "https://github.com/GACLove/ComfyUI-VFI.git"),               # frame interpolation
    ("ComfyUI_Comfyroll_CustomNodes", "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git"),  # CR Float To Integer
    ("ComfyUI-mxToolkit",         "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),       # mxSlider2D (resolution)
    ("ComfyUI-NAG",               "https://github.com/scottmudge/ComfyUI-NAG.git"),            # KSamplerWithNAG (Advanced)
    ("comfyui-adaptiveprompts",   "https://github.com/Alectriciti/comfyui-adaptiveprompts.git"),  # PromptGenerator (the video prompt lands here)
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    # --recurse-submodules: UltimateSDUpscale vendors its upstream repo as a git submodule;
    # an empty submodule folder makes the node import fail. Harmless for the others.
    run(["git", "clone", "--depth", "1", "--recurse-submodules", url, name], f"clone {name}", timeout=180)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## ComfyUI'yi başlat (arka planda)

ComfyUI subprocess olarak kalkar; arayüzün backend'i `QE_COMFY_URL` ile bu adrese konuşur. **90 sn
içinde hazır olmazsa** hücre log'un son 30 satırını basıp durur — sonraki hücre ölü sunucuya
çalışmasın. Tünel yok: ComfyUI'ın kendi arayüzü açılmıyor.

In [ ]:
import subprocess, time, os, urllib.request

# Re-run safety: kill the previous instance before starting a new one
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

# === Start in background (logs to file) ===
# No --enable-manager: nothing opens the UI here, and a missing node already failed loudly during
# install. No tunnel either -- the render cell talks to localhost.
comfy_log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_ROOT, stdout=comfy_log, stderr=subprocess.STDOUT,
)
log(f"ComfyUI başlatıldı (PID {proc.pid}), log: {COMFY_LOG}")

# === Ready? max 90s — otherwise fail-loud with the server's own log ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"{COMFYUI_URL}/system_stats", timeout=2)
        log(f"ComfyUI hazır ({(i + 1) * 2}s)", "OK")
        break
    except Exception:
        pass
else:
    with open(COMFY_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")

In [ ]:
# === Start Flask (background) + cloudflared tunnel ===
# Flask serves the pre-built frontend/dist and /api. It runs as a module from queen-editor/ so
# `backend` resolves as a package. The cell stays OPEN (tail -f): if it ends, Colab calls the
# runtime idle and kills the tunnel. No npm/build here -- the UI ships built (ComfyUI pattern).
import subprocess, time, os, re, urllib.request

APP_DIR = os.path.join(CLONE_DIR, "queen-editor")
FLASK_LOG = "/content/flask.log"

# Re-run safety: kill previous instances before starting new ones
subprocess.run(["pkill", "-f", "backend.main"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(FLASK_LOG, "w")
# The backend reads its Drive root, its ComfyUI address and its xAI key from the environment
# (backend/config.py) -- all three are decided in the cells above, not hardcoded in the app. The
# Civitai cookie is not among them: the app downloads nothing, so it needs no key of its own.
flask_env = {**os.environ, "QE_DRIVE_ROOT": DRIVE_ROOT, "QE_COMFY_URL": COMFYUI_URL,
             "QE_XAI_API_KEY": XAI_API_KEY or ""}
subprocess.Popen(["python", "-m", "backend.main"], cwd=APP_DIR, env=flask_env,
                 stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(FLASK_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Flask 90 sn içinde /api/health'e cevap vermedi — yukarıdaki log'a bak")
print(f"✓ Flask ayakta ({(i + 1) * 2}s)")

if not os.path.isfile("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunlog):
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read())
        if m:
            link = m.group(0)
            break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 Queen Editor: {link}\n")
print("⬆️  Linke gir → projeye tıkla → prompt yaz → Üret.\n")
print("📡 Sunucu çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", FLASK_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — Flask hâlâ arka planda (yeni link için tekrar çalıştır).")